In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [4]:
!pip install Faker psycopg2-binary -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 21.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 64.8 MB/s eta 0:00:00:00:01


In [5]:
from faker import Faker
import random
import json
import psycopg2
from psycopg2.extras import Json

fake = Faker()

sample = {
    "first_name": fake.first_name(),
    "last_name": fake.last_name(),
    "gender": random.choice(["Male", "Female", "Other"]),
    "address": {
        "street": fake.street_address(),
        "city": fake.city(),
        "state": fake.state()
    }
}

sample

{'first_name': 'Edward',
 'last_name': 'Jackson',
 'gender': 'Male',
 'address': {'street': '71093 Curry Mall',
  'city': 'North Jennastad',
  'state': 'Kansas'}}

PostgreSQL SQL and JSON Project in Kaggle
Objective
Create a PostgreSQL database named `testdb`, build a table with varchar and JSON columns, generate 50 synthetic records with Faker, insert them, and validate the results with SQL queries.

In [6]:
!apt-get update -qq
!apt-get install -y postgresql postgresql-contrib -qq

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Preconfiguring packages ...
Selecting previously unselected package logrotate.
(Reading database ... 120314 files and directories currently installed.)
Preparing to unpack .../00-logrotate_3.19.0-1ubuntu1.1_amd64.deb ...
Unpacking logrotate (3.19.0-1ubuntu1.1) ...
Selecting previously unselected package netbase.
Preparing to unpack .../01-netbase_6.3_all.deb ...
Unpacking netbase (6.3) ...
Selecting previously unselected package libcommon-sense-perl:amd64.
Preparing to unpack .../02-libcommon-sense-perl_3.75-2build1_amd64.deb ...
Unpacking libcommon-sense-perl:amd64 (3.75-2build1) ...
Selecting previously unselected package libjson-perl.
Preparing to unpack .../03-libjson-perl_4.04000-1_all.deb ...
Unpacking libjson-perl (4.04000-1) ...
Selecting previously unselected package libtypes-serialiser-perl

In [8]:
!service postgresql start

 * Starting PostgreSQL 14 database server
   ...done.


In [9]:
!sudo -u postgres psql -c "CREATE DATABASE testdb;"

CREATE DATABASE


In [14]:
create_table_cmd = """
sudo -u postgres psql -d testdb -c "
CREATE TABLE customer_profiles (
    last_name VARCHAR(50),
    first_name VARCHAR(50),
    gender VARCHAR(20),
    address JSON
);"
"""

!{create_table_cmd}

CREATE TABLE


In [15]:
verify_table_cmd = 'sudo -u postgres psql -d testdb -c "\\d customer_profiles"'
!{verify_table_cmd}

                  Table "public.customer_profiles"
   Column   |         Type          | Collation | Nullable | Default 
------------+-----------------------+-----------+----------+---------
 last_name  | character varying(50) |           |          | 
 first_name | character varying(50) |           |          | 
 gender     | character varying(20) |           |          | 
 address    | json                  |           |          | 



In [18]:
!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'postgres';"

ALTER ROLE


In [19]:
from faker import Faker
import random
import psycopg2
from psycopg2.extras import Json

fake = Faker()

conn = psycopg2.connect(
    dbname="testdb",
    user="postgres",
    password="postgres",
    host="localhost",
    port="5432"
)

cur = conn.cursor()

for _ in range(50):
    first_name = fake.first_name()
    last_name = fake.last_name()
    gender = random.choice(["Male", "Female", "Other"])
    address = {
        "street": fake.street_address(),
        "city": fake.city(),
        "state": fake.state()
    }

    cur.execute(
        """
        INSERT INTO customer_profiles (last_name, first_name, gender, address)
        VALUES (%s, %s, %s, %s)
        """,
        (last_name, first_name, gender, Json(address))
    )

conn.commit()
cur.close()
conn.close()

print("50 records inserted successfully.")

50 records inserted successfully.


In [20]:
conn = psycopg2.connect(
    dbname="testdb",
    user="postgres",
    password="postgres",
    host="localhost",
    port="5432"
)

cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM customer_profiles;")
count = cur.fetchone()[0]

cur.close()
conn.close()

print("Total rows:", count)

Total rows: 50


In [21]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    dbname="testdb",
    user="postgres",
    password="postgres",
    host="localhost",
    port="5432"
)

query1 = "SELECT * FROM customer_profiles LIMIT 5;"
df1 = pd.read_sql_query(query1, conn)
df1

/tmp/ipykernel_55/1565201631.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df1 = pd.read_sql_query(query1, conn)


,last_name,first_name,gender,address
0,Allen,Brian,Male,"{'street': '7634 Matthew Cape', 'city': 'North..."
1,Torres,Joseph,Female,"{'street': '235 Gregory Lake', 'city': 'Christ..."
2,Myers,Andrew,Female,"{'street': '72420 Annette Skyway', 'city': 'Jo..."
3,Rhodes,Heather,Male,"{'street': '89193 Jennifer Dale Suite 505', 'c..."
4,Maldonado,John,Male,"{'street': '88854 Clark Locks Suite 123', 'cit..."


In [22]:
query2 = """
SELECT 
    first_name,
    last_name,
    gender,
    address->>'city' AS city,
    address->>'state' AS state
FROM customer_profiles
LIMIT 10;
"""

df2 = pd.read_sql_query(query2, conn)
df2

/tmp/ipykernel_55/3140716489.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df2 = pd.read_sql_query(query2, conn)


,first_name,last_name,gender,city,state
0,Brian,Allen,Male,North Melaniechester,Delaware
1,Joseph,Torres,Female,Christiebury,North Carolina
2,Andrew,Myers,Female,Johnstonton,Wyoming
3,Heather,Rhodes,Male,New Christopherstad,Vermont
4,John,Maldonado,Male,Gracemouth,Maine
5,Sean,Smith,Female,Michaelland,Oklahoma
6,Veronica,Buchanan,Male,Jeremychester,Louisiana
7,Daniel,Horn,Female,Scottmouth,Utah
8,Cassie,Stevenson,Female,West Ronniebury,Kentucky
9,Michael,Hamilton,Other,Jamesland,Arkansas


In [23]:
query3 = """
SELECT 
    first_name,
    last_name,
    address->>'state' AS state
FROM customer_profiles
WHERE address->>'state' = 'California';
"""

df3 = pd.read_sql_query(query3, conn)
df3

/tmp/ipykernel_55/1103024478.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df3 = pd.read_sql_query(query3, conn)


,first_name,last_name,state


In [24]:
query4 = """
SELECT gender, COUNT(*) AS total
FROM customer_profiles
GROUP BY gender
ORDER BY total DESC;
"""

df4 = pd.read_sql_query(query4, conn)
df4

/tmp/ipykernel_55/3744468453.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df4 = pd.read_sql_query(query4, conn)


,gender,total
0,Female,22
1,Other,15
2,Male,13


In [25]:
conn.close()
print("Validation queries completed.")

Validation queries completed.


References

Databricks. (n.d.). *What is Delta Lake?* https://docs.databricks.com/en/delta/index.html

GeeksforGeeks. (2024, March 7). *Difference between SQL and NoSQL*. https://www.geeksforgeeks.org/difference-between-sql-and-nosql/

Passador, S. (2021, December 16). *Docker compose with Python and PostgreSQL*. Medium. https://stefanopassador.medium.com/docker-compose-with-python-and-posgresql-45c4c5174299

PostgreSQL Global Development Group. (n.d.). *PostgreSQL documentation*. https://www.postgresql.org/docs/

PostgreSQL Tutorial. (2024, April 21). *PostgreSQL tutorial*. https://www.postgresqltutorial.com/